In [1]:
# Fixed XSS & SQLi Detection Training Notebook

# 0) Mount GPU check
!nvidia-smi || echo "No GPU detected (make sure Runtime -> Change runtime type -> GPU)"

# 1) Install packages
!pip install -q kaggle==1.5.16 datasets transformers accelerate evaluate peft datasets[s3] scikit-learn seqeval

# 2) Upload kaggle.json
from google.colab import files
uploaded = files.upload()

# 3) Extract datasets
import zipfile
import os

os.makedirs("datasets/xss", exist_ok=True)
os.makedirs("datasets/sqli", exist_ok=True)

with zipfile.ZipFile("/content/XSS_dataset.csv.zip", "r") as zip_ref:
    zip_ref.extractall("datasets/xss")

with zipfile.ZipFile("/content/SQLiV3.csv.zip", "r") as zip_ref:
    zip_ref.extractall("datasets/sqli")

# 4) Load datasets
import pandas as pd
from pathlib import Path

sqli_path = Path("/content/datasets/sqli/SQLiV3.csv")
xss_path = Path("/content/datasets/xss/XSS_dataset.csv")

def try_load_csv(path):
    try:
        return pd.read_csv(path, encoding="utf-8", low_memory=False)
    except:
        try:
            return pd.read_csv(path, encoding="latin1", low_memory=False)
        except Exception as e:
            print(f"❌ Failed to load {path}: {e}")
            return None

sqli_df = try_load_csv(sqli_path)
xss_df = try_load_csv(xss_path)

print("✅ SQLi dataset shape:", None if sqli_df is None else sqli_df.shape)
print("✅ XSS dataset shape:", None if xss_df is None else xss_df.shape)

# 5) Clean and standardize data
def clean_and_select(df, name):
    df.columns = [c.strip().lower() for c in df.columns]

    text_col = next((c for c in df.columns if any(k in c for k in ["sentence","text","payload","query","content","body"])), None)
    label_col = next((c for c in df.columns if "label" in c), None)

    if not text_col or not label_col:
        print(f"⚠️ Couldn't find proper columns in {name}")
        print("Columns found:", df.columns)
        return None

    df = df[[text_col, label_col]].rename(columns={text_col: "sentence", label_col: "label"})
    df = df[df["label"].isin([0, 1, "0", "1"])].copy()
    df["label"] = df["label"].astype(int)
    df["sentence"] = df["sentence"].astype(str).str.strip()
    df.dropna(subset=["sentence", "label"], inplace=True)
    df["dataset"] = name.lower()
    return df

sqli_df = clean_and_select(sqli_df, "SQLi")
xss_df = clean_and_select(xss_df, "XSS")

combined_df = pd.concat([df for df in [sqli_df, xss_df] if df is not None], ignore_index=True)

# 6) Create attack_type column
def assign_attack_type(row):
    if row["label"] == 1:
        if row["dataset"] == "sqli":
            return "sqli"
        elif row["dataset"] == "xss":
            return "xss"
    return "normal"

combined_df["attack_type"] = combined_df.apply(assign_attack_type, axis=1)

print(f"\n✅ Combined dataset shape: {combined_df.shape}")
print(combined_df["attack_type"].value_counts())

# 7) Create multi-class labels
def map_to_multiclass(row):
    if row["attack_type"] == "normal":
        return 0
    if row["attack_type"] == "sqli":
        return 1
    if row["attack_type"] == "xss":
        return 2
    return 0 if row["label"]==0 else 1

combined_df["multi_label"] = combined_df.apply(map_to_multiclass, axis=1)
combined_df = combined_df[["sentence","multi_label","attack_type"]].rename(columns={"multi_label":"label"})

print("\n✅ Label distribution:")
print(combined_df["label"].value_counts())

# 8) Create HuggingFace dataset with stratified splits
from datasets import Dataset, DatasetDict, ClassLabel

df = combined_df.copy()

# Normalize labels
label_map = {"normal": 0, "sqli": 1, "xss": 2}
df["attack_type"] = df["attack_type"].str.lower().str.strip()
df["label"] = df["attack_type"].map(label_map)
df = df[df["label"].isin([0, 1, 2])].reset_index(drop=True)

# Remove any remaining None/NaN values
df = df.dropna(subset=["sentence", "label"])
df["label"] = df["label"].astype(int)

print("\n✅ Final label distribution:")
print(df["label"].value_counts())

# Create HF dataset
hf_full = Dataset.from_pandas(df[["sentence", "label"]].rename(columns={"sentence": "text"}), preserve_index=False)

# Convert to ClassLabel
labels_feature = ClassLabel(names=["normal", "sqli", "xss"])
hf_full = hf_full.cast_column("label", labels_feature)

# Stratified splits
seed = 42
ds_train_val_test = hf_full.train_test_split(test_size=0.10, stratify_by_column="label", seed=seed)
train_and_val = ds_train_val_test["train"].train_test_split(test_size=0.1111111, stratify_by_column="label", seed=seed)

dataset = DatasetDict({
    "train": train_and_val["train"],
    "validation": train_and_val["test"],
    "test": ds_train_val_test["test"]
})

print("\n✅ Dataset splits:")
print(dataset)

# Save to disk
out_dir = "/content/hf_attack_dataset"
dataset.save_to_disk(out_dir)
print(f"\n💾 Saved dataset to {out_dir}")

# 9) Load model and tokenizer
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification

tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=128)

tokenized_dataset = dataset.map(tokenize, batched=True)

# Verify no None labels
for split in ["train", "validation", "test"]:
    none_count = sum(1 for x in tokenized_dataset[split]["label"] if x is None)
    print(f"None labels in {split}: {none_count}")

num_labels = 3
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_labels,
    id2label={0:"normal", 1:"sqli", 2:"xss"},
    label2id={"normal":0, "sqli":1, "xss":2}
)

# 10) Apply LoRA
from peft import LoraConfig, get_peft_model, TaskType
import torch

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_lin","v_lin"]
)

model = get_peft_model(model, lora_config)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# 11) Define metrics - FIXED VERSION
import numpy as np
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, classification_report

def compute_metrics(eval_pred):
    """Fixed compute_metrics that handles None values properly"""
    logits, labels = eval_pred

    # Remove None values if any
    valid_indices = [i for i, label in enumerate(labels) if label is not None]

    if len(valid_indices) < len(labels):
        print(f"⚠️ Warning: Found {len(labels) - len(valid_indices)} None labels")
        logits = logits[valid_indices]
        labels = [labels[i] for i in valid_indices]

    labels = np.array(labels, dtype=int)
    preds = np.argmax(logits, axis=-1)

    # Compute metrics
    f1 = f1_score(labels, preds, average="macro")
    acc = accuracy_score(labels, preds)
    prec = precision_score(labels, preds, average="macro")
    rec = recall_score(labels, preds, average="macro")

    # Per-class report
    report = classification_report(labels, preds, output_dict=True, target_names=["normal", "sqli", "xss"])

    return {
        "accuracy": acc,
        "eval_f1": f1,
        "precision": prec,
        "recall": rec,
        "normal_f1": report["normal"]["f1-score"],
        "sqli_f1": report["sqli"]["f1-score"],
        "xss_f1": report["xss"]["f1-score"]
    }

# 12) Training setup
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding
import os

os.environ["WANDB_DISABLED"] = "true"

data_collator = DataCollatorWithPadding(tokenizer)

training_args = TrainingArguments(
    output_dir="/content/model_output",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    logging_steps=50,
    save_steps=200,
    save_total_limit=2,
    learning_rate=2e-5,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1",
    report_to=[]
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# 13) Train
print("\n🚀 Starting training...")
trainer.train()


Wed Oct 15 06:51:01 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   67C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Saving SQLiV3.csv.zip to SQLiV3.csv.zip
Saving XSS_dataset.csv.zip to XSS_dataset.csv.zip
✅ SQLi dataset shape: (30919, 4)
✅ XSS dataset shape: (13686, 3)

✅ Combined dataset shape: (44295, 4)
attack_type
normal    25581
sqli      11341
xss        7373
Name: count, dtype: int64

✅ Label distribution:
label
0    25581
1    11341
2     7373
Name: count, dtype: int64

✅ Final label distribution:
label
0    25581
1    11341
2     7373
Name: count, dtype: int64


Casting the dataset:   0%|          | 0/44295 [00:00<?, ? examples/s]


✅ Dataset splits:
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 35435
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 4430
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 4430
    })
})


Saving the dataset (0/1 shards):   0%|          | 0/35435 [00:00<?, ? examples/s]

...

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



🚀 Starting training...


Epoch,Training Loss,Validation Loss,F1,Accuracy,Precision,Recall,Normal F1,Sqli F1,Xss F1
1,0.048100,0.031216,0.991240,0.992325,0.994274,0.988289,0.994163,0.989809,0.989747
2,0.024700,0.023298,0.993483,0.994357,0.995770,0.991244,0.995715,0.992920,0.991814
3,0.014900,0.020166,0.994253,0.995260,0.996124,0.992419,0.996686,0.994260,0.991814


TrainOutput(global_step=6645, training_loss=0.05756496735494598, metrics={'train_runtime': 298.5755, 'train_samples_per_second': 356.041, 'train_steps_per_second': 22.256, 'total_flos': 3580993727009280.0, 'train_loss': 0.05756496735494598, 'epoch': 3.0})

In [3]:

# 14) Evaluate on test set
print("\n📊 Evaluating on test set...")
test_metrics = trainer.evaluate(eval_dataset=tokenized_dataset["test"])

print("\n" + "="*50)
print("TEST SET RESULTS")
print("="*50)
print(f"Accuracy:   {test_metrics['eval_accuracy']:.4f}")
print(f"Precision:  {test_metrics['eval_precision']:.4f}")
print(f"Recall:     {test_metrics['eval_recall']:.4f}")
print("\nPer-class F1 scores:")
print(f"  Normal:   {test_metrics['eval_normal_f1']:.4f}")
print(f"  SQLi:     {test_metrics['eval_sqli_f1']:.4f}")
print(f"  XSS:      {test_metrics['eval_xss_f1']:.4f}")
print("="*50)





📊 Evaluating on test set...



TEST SET RESULTS
Accuracy:   0.9953
Precision:  0.9956
Recall:     0.9932

Per-class F1 scores:
  Normal:   0.9963
  SQLi:     0.9952
  XSS:      0.9918


In [4]:
# 15) Save model
model_path = "/content/model_output/final_lora_model"
model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)
print(f"\n💾 Saved model to {model_path}")




💾 Saved model to /content/model_output/final_lora_model


In [5]:
# 16) Test inference
def predict_attack(text):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)
        pred = torch.argmax(probs, dim=-1).item()

    labels = {0: "normal", 1: "sqli", 2: "xss"}
    return labels[pred], probs[0].cpu().numpy()

# Test samples
test_samples = [
    "SELECT * FROM users WHERE id=1",
    "' OR '1'='1' --",
    "<script>alert('XSS')</script>",
    "Hello world"
]
print("\n🔍 Testing predictions:")
for sample in test_samples:
    pred, probs = predict_attack(sample)
    print(f"\nText: {sample}")
    print(f"Prediction: {pred}")
    print(f"Confidence: normal={probs[0]:.3f}, sqli={probs[1]:.3f}, xss={probs[2]:.3f}")


🔍 Testing predictions:

Text: SELECT * FROM users WHERE id=1
Prediction: normal
Confidence: normal=0.989, sqli=0.010, xss=0.000

Text: ' OR '1'='1' --
Prediction: sqli
Confidence: normal=0.000, sqli=1.000, xss=0.000

Text: <script>alert('XSS')</script>
Prediction: xss
Confidence: normal=0.000, sqli=0.000, xss=1.000

Text: Hello world
Prediction: normal
Confidence: normal=0.998, sqli=0.001, xss=0.001


In [6]:
!zip -r final_lora_model.zip /content/model_output/final_lora_model


  adding: content/model_output/final_lora_model/ (stored 0%)
  adding: content/model_output/final_lora_model/special_tokens_map.json (deflated 42%)
  adding: content/model_output/final_lora_model/adapter_config.json (deflated 55%)
  adding: content/model_output/final_lora_model/README.md (deflated 66%)
  adding: content/model_output/final_lora_model/vocab.txt (deflated 53%)
  adding: content/model_output/final_lora_model/tokenizer.json (deflated 71%)
  adding: content/model_output/final_lora_model/adapter_model.safetensors (deflated 7%)
  adding: content/model_output/final_lora_model/tokenizer_config.json (deflated 75%)
